# HyperSense — External Transportability

## Ghana → Nigeria

### Objective

Evaluate how well the existing HyperSense v1.1 model generalizes to an independent Nigerian clinical cohort from the REMAH study, **without retraining or modifying the model using Nigerian data**.

### Research Question

> **Can the Ghana-derived HyperSense v1.1 model generalize to Nigerian adults without retraining?**

### Existing Model

HyperSense v1.1 is an XGBoost model developed using Ghana DHS 2014 with six predictors:

- Age
- Sex
- Residence
- Educational level
- Tobacco use
- BMI

### External Dataset

REMAH provides Nigerian clinical and behavioural data, including five clinic BP readings, anthropometry, tobacco use, pulse rate and laboratory measurements.

Only variables genuinely comparable with the v1.1 predictors will be used for transportability evaluation.

| HyperSense v1.1 | REMAH |
|---|---|
| `age` | `AGE` |
| `gender` | `SEX` |
| `residence` | `SITE` |
| `educational_level` | `EDU_QUA` |
| `tobacco_use` | `SMK_C` / `SMKL` |
| `bmi` | derived from `HT` and `WT` |

### Outcome

Hypertension will use the measurement-only definition:

> **Mean SBP ≥140 mmHg and/or mean DBP ≥90 mmHg.**

Mean BP will be derived from the five clinic BP readings. `HTN_DR` (Hypertensive Drug Therapy) will not be included in the primary transportability outcome because an equivalent treatment variable was unavailable in the DHS data with which the Ghanaian model was built.

### Evaluation

Primary evaluation will include:

- ROC-AUC
- Sensitivity and specificity
- PPV and NPV
- Calibration intercept and slope
- Brier score
- Calibration plot

The original v1.1 decision threshold will be evaluated without re-optimizing it on REMAH.

### Workflow

- [ ] Load and inspect REMAH
- [ ] Verify data quality and coding
- [ ] Construct the transportability outcome
- [ ] Derive BMI
- [ ] Harmonize the six predictors
- [ ] Verify compatibility with the v1.1 model
- [ ] Apply the existing v1.1 model
- [ ] Evaluate discrimination
- [ ] Evaluate calibration
- [ ] Perform age-range sensitivity analysis
- [ ] Summarize transportability findings
- [ ] Document limitations and implications

> KEY PRICNIPLE: **Validate first. Adapt second.**

In [25]:
# IMPORT REQUIRED LIBRARIES

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

# Machine learning / evaluation
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

# Model loading
import joblib

# File and path management
from pathlib import Path

#### 1. Load Dataset

In [26]:
remah = pd.read_excel("../data/REMAH/remah-1.xlsx")

print(f"Shape: {remah.shape}")
remah.head()

Shape: (4187, 76)


,S_NO,PIN,SEX,EST_AGE,DOB,AGE,SITE,EDU_QUA,M_STATUS,W_STATUS,SMK_C,SMK_D,SMK_D1,SMK_D2,M_CGRT,HR_CGRT,PIPE,CIGAR,SMK_OTHERS,SMK_P,SMK_P1,SMK_P2,SMKL,SMKL_MO,SMKL_NO,SMKL_CHEW,SMKL_OTHERS,SMK_PA_H,SMK_PA_WO,DRK,DRK_12,DRK12_FREQ,DRK_30,WRK_VI,WRK_VI_D,WRK_VI_T,WRK_MI,WRK_MI_D,WRK_MI_T,PA_W,PA_W_D,PA_W_T,SPRT_VI,SPRT_VI_D,SPRT_VI_T,SPRT_MI,SPRT_MI_D,SPRT_MI_T,HTN,HTN_DR,DM_M,DM,DM_INS,DM_OHA,CSBP1,CDBP1,CSBP2,CDBP2,CSBP3,CDBP3,CSBP4,CDBP4,CSBP5,CDBP5,PR,HT,WT,WC,HC,BLD_GLU,NA_URINE,HDL_CHOL,LDL_CHOL,TAG,TO_CHOL,K_URINE
0,1,10011001,1,78.000,1939-06-01,77.877,1,2.000,5.000,7.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,NaN,NaN,1.000,NaN,5.000,NaN,NaN,2.000,77.000,1.000,1.000,4.000,1.000,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,NaN,NaN,NaN,2.000,NaN,NaN,NaN,128.000,70.000,128.000,68.000,118.000,64.000,120.000,70.000,120.000,60.000,81.000,177.000,59.000,78.000,85.000,116.000,NaN,NaN,NaN,NaN,NaN,NaN
1,2,11001002,1,NaN,1964-02-02,53.197,2,3.000,2.000,3.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,NaN,NaN,2.000,NaN,NaN,NaN,NaN,77.000,77.000,1.000,1.000,NaN,1.000,2.000,NaN,NaN,1.000,3.000,00:10:00,1.000,7.000,00:15:00,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,1.000,2.000,NaN,NaN,120.000,78.000,120.000,80.000,124.000,82.000,130.000,84.000,128.000,82.000,70.000,177.000,95.000,114.000,116.000,138.000,NaN,NaN,NaN,NaN,NaN,NaN
2,3,10035001,1,NaN,1955-01-01,62.288,1,4.000,2.000,3.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,30.000,32.000,2.000,NaN,NaN,NaN,NaN,4.000,77.000,1.000,1.000,3.000,1.000,1.000,6.000,08:00:00,2.000,NaN,NaN,1.000,7.000,02:15:00,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,2.000,NaN,NaN,NaN,120.000,68.000,122.000,78.000,120.000,80.000,120.000,70.000,120.000,78.000,60.000,168.000,55.000,83.000,78.000,80.000,NaN,NaN,NaN,NaN,NaN,NaN
3,4,10001011,1,67.000,1950-01-01,67.288,1,2.000,2.000,7.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,77.000,NaN,2.000,NaN,NaN,NaN,NaN,77.000,77.000,1.000,1.000,3.000,1.000,2.000,NaN,NaN,2.000,NaN,NaN,1.000,7.000,00:30:00,2.000,NaN,NaN,2.000,NaN,NaN,2.000,NaN,2.000,2.000,NaN,NaN,138.000,70.000,130.000,76.000,128.000,70.000,128.000,76.000,130.000,70.000,63.000,165.000,55.000,80.000,89.000,131.000,NaN,NaN,NaN,NaN,NaN,NaN
4,5,10001001,1,NaN,1982-07-10,34.734,1,5.000,2.000,1.000,2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,33.000,1.000,2.000,NaN,NaN,NaN,NaN,2.000,2.000,1.000,1.000,3.000,1.000,2.000,NaN,NaN,2.000,NaN,NaN,1.000,7.000,00:20:00,2.000,NaN,NaN,1.000,1.000,01:00:00,2.000,NaN,1.000,2.000,NaN,NaN,142.000,84.000,138.000,86.000,136.000,86.000,134.000,82.000,138.000,78.000,70.000,189.500,105.000,104.000,115.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Dataset Characteristics

In [29]:
# Initial dataset inspection

print("Shape:", remah.shape)
print("\nColumns:")
print(remah.columns.tolist())

Shape: (4187, 76)

Columns:
['S_NO', 'PIN', 'SEX', 'EST_AGE', 'DOB', 'AGE', 'SITE', 'EDU_QUA', 'M_STATUS', 'W_STATUS', 'SMK_C', 'SMK_D', 'SMK_D1', 'SMK_D2', 'M_CGRT', 'HR_CGRT', 'PIPE', 'CIGAR', 'SMK_OTHERS', 'SMK_P', 'SMK_P1', 'SMK_P2', 'SMKL', 'SMKL_MO', 'SMKL_NO', 'SMKL_CHEW', 'SMKL_OTHERS', 'SMK_PA_H', 'SMK_PA_WO', 'DRK', 'DRK_12', 'DRK12_FREQ', 'DRK_30', 'WRK_VI', 'WRK_VI_D', 'WRK_VI_T', 'WRK_MI', 'WRK_MI_D', 'WRK_MI_T', 'PA_W', 'PA_W_D', 'PA_W_T', 'SPRT_VI', 'SPRT_VI_D', 'SPRT_VI_T', 'SPRT_MI', 'SPRT_MI_D', 'SPRT_MI_T', 'HTN', 'HTN_DR', 'DM_M', 'DM', 'DM_INS', 'DM_OHA', 'CSBP1', 'CDBP1', 'CSBP2', 'CDBP2', 'CSBP3', 'CDBP3', 'CSBP4', 'CDBP4', 'CSBP5', 'CDBP5', 'PR', 'HT', 'WT', 'WC', 'HC', 'BLD_GLU', 'NA_URINE', 'HDL_CHOL', 'LDL_CHOL', 'TAG', 'TO_CHOL', 'K_URINE']


In [70]:
# Data types of all variables

print("\nData types:")
display(remah.dtypes.to_frame("dtype"))


Data types:


,dtype
S_NO,int64
PIN,int64
SEX,int64
EST_AGE,float64
DOB,datetime64[ns]
AGE,float64
SITE,int64
EDU_QUA,float64
M_STATUS,float64
W_STATUS,float64


In [76]:
# Missing values?

missing_summary = (
    remah.isna()
    .sum()
    .to_frame("Missing")
    .assign(
        Missing_pct=lambda x: (x["Missing"] / len(remah) * 100).round(2)
    )
    .assign(
        Complete=lambda x: x["Missing"] == 0
    )
    .sort_values("Missing", ascending=False)
)
print("\nMissing values:")
display(missing_summary)


Missing values:


,Missing,Missing_pct,Complete
PIPE,4187,100.000,False
SMKL_CHEW,4182,99.880,False
SMKL_OTHERS,4180,99.830,False
SMK_OTHERS,4178,99.790,False
HR_CGRT,4174,99.690,False
SMKL_MO,4168,99.550,False
CIGAR,4160,99.360,False
SMKL_NO,4121,98.420,False
M_CGRT,4103,97.990,False
SMK_D2,4091,97.710,False


In [78]:
# Inspect the coding and frequency of key categorical variables
    
cat_variables = ["SEX", "SITE", "EDU_QUA", "SMK_C", "SMKL", "HTN", "HTN_DR"]

table = []

for var in cat_variables:
    counts = remah[var].value_counts(dropna=False).sort_index()
    
    for category, frequency in counts.items():
        table.append({
            "Variable": var,
            "Category": category,
            "Frequency": frequency
        })

freq_table = pd.DataFrame(table)

print("Categorical variables:")
display(freq_table)

Categorical variables:


,Variable,Category,Frequency
0,SEX,1.000,1814
1,SEX,2.000,2373
2,SITE,1.000,2171
3,SITE,2.000,2016
4,EDU_QUA,1.000,658
5,EDU_QUA,2.000,193
6,EDU_QUA,3.000,707
7,EDU_QUA,4.000,1158
8,EDU_QUA,5.000,1131
9,EDU_QUA,7.000,299


In [79]:
# Inspect ranges and potential anomalies in continuous transportability variables

continuous_vars = [
    "AGE",
    "HT",
    "WT",
    "CSBP1", "CSBP2", "CSBP3", "CSBP4", "CSBP5",
    "CDBP1", "CDBP2", "CDBP3", "CDBP4", "CDBP5",
]

# Summarize distribution, missingness, and range
quality_summary = (
    remah[continuous_vars]
    .describe()
    .T
    .assign(
        missing=remah[continuous_vars].isna().sum(),
        missing_pct=(remah[continuous_vars].isna().mean() * 100).round(2)
    )
    .sort_values(by="missing", ascending=False)
)

print("Continuous variables:")
display(quality_summary)

Continuous variables:


,count,mean,std,min,25%,50%,75%,max,missing,missing_pct
WT,4163.000,65.348,14.779,30.000,55.000,63.000,74.000,136.000,24,0.570
HT,4164.000,163.419,8.793,131.000,157.000,163.000,169.125,195.000,23,0.550
CDBP5,4169.000,78.254,13.591,22.000,70.000,78.000,86.000,168.000,18,0.430
CSBP5,4171.000,125.562,23.337,64.000,110.000,122.000,138.000,254.000,16,0.380
CDBP1,4182.000,78.027,13.606,40.000,70.000,78.000,86.000,162.000,5,0.120
CSBP2,4183.000,126.546,23.869,64.000,110.000,122.000,140.000,252.000,4,0.100
CSBP1,4183.000,127.035,24.218,60.000,110.000,122.000,140.000,250.000,4,0.100
AGE,4183.000,43.835,16.298,14.510,30.660,41.959,55.142,98.792,4,0.100
CSBP4,4183.000,125.719,23.366,66.000,110.000,122.000,138.000,256.000,4,0.100
CDBP4,4183.000,78.238,13.561,40.000,70.000,76.000,86.000,168.000,4,0.100
